In [18]:

from scraper import fetch_website_links
from openai import OpenAI
import gradio as gr

In [2]:
ollama_url = "http://localhost:11434/v1"
MODEL="llama3.2"
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

In [3]:

# Again this is typical Experimental mindset - I'm changing the global variable we used above:

system_message = """
You are an assistant that analyzes the contents of a company website landing page
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
"""

In [11]:
def stream_ollama(prompt):
    messages=[{"role": "system", "content": system_message}, {"role": "user", "content": prompt}]
    stream= ollama.chat.completions.create(
        model=MODEL,
        messages=messages,
        stream=True
    )

    response = ""
    for chunk in stream:
        response+= chunk.choices[0].delta.content or ""
    yield response


 


In [19]:
def stream_brochure(company_name, url, model):
    yield ""
    prompt = f"Please generate a company brochure for {company_name}. Here is their landing page:\n"
    links = fetch_website_links(url)
    prompt += "\n".join(links)
    if model=="ollama":
        result = stream_ollama(prompt)
    else:
        raise ValueError("Unknown model")
    yield from result

In [21]:
name_input = gr.Textbox(label="Company name:")
url_input = gr.Textbox(label="Landing page URL including http:// or https://")
model_selector = gr.Dropdown(["qwen", "ollama"], label="Select model", value="ollama")
message_output = gr.Markdown(label="Response:")
view = gr.Interface(
    fn=stream_brochure,
    title="LLM",
    inputs=[name_input, url_input, model_selector],
    outputs=[message_output],
    examples=[
         ["Hugging Face", "https://huggingface.co"],
         ["Edward Donner", "https://edwarddonner.com"],
         ["Supreme Cryogenic", "https://supremecryogenic.com/"]
     ],
     flagging_mode="never"
)

view.launch()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.
